# 01 — Screen & Build Combined Universe

Uses `yfscreen` to systematically construct investable universes across multiple sectors,
then combines them into a single universe for cross-sector clustering.

**Pipeline:**
1. Explore available yfscreen filter fields
2. Define screening criteria (liquidity, fundamentals)
3. Run screen for each target sector
4. Combine all sectors into one universe with sector map
5. Fetch prices and apply data quality filters
6. Cache combined universe to `screener/data/combined/`

In [ ]:
import sys, os

# Project root must come FIRST on sys.path so that 'config' resolves
# to the project-level config.py, not screener/config.py.
project_root = os.path.abspath(os.path.join('..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfscreen as yfs

from screener.config import ScreenerConfig
from screener.screening import (
    explore_available_filters, screen_combined_universe,
)
from screener.universe import fetch_and_validate_prices, cache_universe

%matplotlib inline
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

## 1. Explore Available yfscreen Filters

Confirm the exact field names for market cap, volume, EBITDA, etc.

In [ ]:
filters = explore_available_filters()

## 2. Configure Screening Parameters

In [ ]:
cfg = ScreenerConfig()
print("Screening config:")
for field in cfg.__dataclass_fields__:
    print(f"  {field}: {getattr(cfg, field)}")

## 3. Screen All Sectors & Build Combined Universe

In [ ]:
combined_tickers, sector_map, per_sector = screen_combined_universe(cfg)

print(f"\nTotal tickers (incl ^GSPC): {len(combined_tickers)}")
print(f"Sector map entries: {len(sector_map)}")

# Show sector composition
sector_counts = pd.Series(sector_map).value_counts()
print("\nSector composition:")
for sector, count in sector_counts.items():
    print(f"  {sector}: {count}")

## 4. Fetch Prices & Apply Data Quality Filters

In [ ]:
print(f"Fetching prices for {len(combined_tickers)} tickers...")
prices, dropped = fetch_and_validate_prices(combined_tickers, cfg)

kept_tickers = [c for c in prices.columns if c != '^GSPC']
print(f"\nKept: {len(kept_tickers)} tickers")
print(f"Dropped: {len(dropped)} tickers")

if dropped:
    for ticker, reason in dropped[:10]:
        print(f"  {ticker}: {reason}")
    if len(dropped) > 10:
        print(f"  ... and {len(dropped) - 10} more")

# Update sector_map to only include kept tickers
sector_map = {t: s for t, s in sector_map.items() if t in kept_tickers}
print(f"\nSector map after filtering: {len(sector_map)} tickers")

sector_counts = pd.Series(sector_map).value_counts()
for sector, count in sector_counts.items():
    print(f"  {sector}: {count}")

## 5. Data Quality Report

In [ ]:
coverage = prices[kept_tickers].notna().mean()

print(f"Price matrix: {prices.shape[0]} timestamps x {len(kept_tickers)} tickers")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")
print(f"Avg coverage: {coverage.mean():.1%}")
print(f"Min coverage: {coverage.min():.1%}")
print(f"Median coverage: {coverage.median():.1%}")

# Per-sector quality
for sector in sorted(set(sector_map.values())):
    stickers = [t for t, s in sector_map.items() if s == sector]
    scov = prices[stickers].notna().mean()
    print(f"\n  {sector} ({len(stickers)} tickers):")
    print(f"    Avg coverage: {scov.mean():.1%}, Min: {scov.min():.1%}")

## 6. Cache Combined Universe

In [ ]:
# Cache with sector_map so downstream notebooks can classify pairs
cache_universe('combined', kept_tickers + ['^GSPC'], prices, sector_map=sector_map)
print("Combined universe cached to screener/data/combined/")

## 7. Universe Composition Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sector distribution
sector_counts = pd.Series(sector_map).value_counts()
sector_counts.plot(kind='bar', ax=axes[0])
axes[0].set_title('Tickers per Sector')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Coverage distribution
coverage.hist(bins=20, ax=axes[1])
axes[1].axvline(x=cfg.min_data_coverage, color='r', linestyle='--', label=f'Min threshold ({cfg.min_data_coverage:.0%})')
axes[1].set_title('Data Coverage Distribution')
axes[1].set_xlabel('Coverage fraction')
axes[1].legend()

plt.tight_layout()
plt.show()